# Gradients and fitting — documentation examples

Companion notebook to the **Gradients and fitting** documentation page
(`docs/source/gradients.rst`). One section per code snippet.

Requires the core `stellar-spice` install; the fit uses `optax` (`pip install optax`).
Runs on a laptop CPU in a few minutes.

In [1]:
%matplotlib inline
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
os.environ.setdefault("JAX_ENABLE_X64", "1")   # finite-difference checks below need float64

import time
from pathlib import Path
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

DOCS_IMG = Path("..") / ".." / "docs" / "img"

def save_doc_fig(name, make_fig):
    """Save make_fig() as <name>.png (light) and <name>_dark.png in docs/img."""
    for style, suffix in [("default", ""), ("dark_background", "_dark")]:
        with plt.style.context(style):
            fig = make_fig()
            fig.savefig(DOCS_IMG / f"{name}{suffix}.png", dpi=120, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            plt.close(fig)
    print(f"saved {name}.png / {name}_dark.png")

T_START = time.perf_counter()

## A magnitude and its derivative

In [2]:
from spice.models import IcosphereModel
from spice.models.mesh_transform import update_parameter
from spice.spectrum import Blackbody, simulate_observed_flux, AB_passband_luminosity
from spice.spectrum.filter import JohnsonCousinsV

bb = Blackbody()
star = IcosphereModel.construct(1000, 1., 1., bb.solar_parameters, bb.parameter_names)
wl = jnp.linspace(4500., 7000., 500)
V = JohnsonCousinsV()

def v_mag(teff):
    m = update_parameter(star, 'Teff', jnp.full(star.parameters.shape[0], teff),
                         bb.parameter_names)
    flux = simulate_observed_flux(bb.intensity, m, jnp.log10(wl),
                                  disable_doppler_shift=True)[:, 0]
    return AB_passband_luminosity(V, wl, flux)

mag, dmag = jax.value_and_grad(v_mag)(5777.)
fd = (v_mag(5778.) - v_mag(5776.)) / 2.
print(f"V = {mag:.4f} mag, dV/dTeff = {dmag:.4e} mag/K (finite difference {fd:.4e})")

PHOEBE: passband "Bolometric:900-40000" has a newer version available.  Run phoebe.list_passband_online_history("Bolometric:900-40000") to get a list of available changes and phoebe.update_passband("Bolometric:900-40000") or phoebe.update_all_passbands() to update.


[spice] IcosphereModel constructed in 1.6 s


V = 4.8668 mag, dV/dTeff = -8.6262e-04 mag/K (finite difference -8.6262e-04)


## A line profile and its Jacobian

In [3]:
from spice.models.mesh_transform import add_rotation, evaluate_rotation
from spice.spectrum import GaussianLineEmulator

gl = GaussianLineEmulator(line_centers=[5500.], line_widths=[0.3], line_depths=[0.5])
star_gl = IcosphereModel.construct(1000, 1., 1., gl.to_parameters(), gl.parameter_names)
wl_line = jnp.linspace(5498., 5502., 200)

def normalised_profile(vrot):
    m = evaluate_rotation(add_rotation(star_gl, vrot), 0.)
    f = simulate_observed_flux(gl.intensity, m, jnp.log10(wl_line))
    return f[:, 0] / f[:, 1]

J = jax.jacfwd(normalised_profile)(20.)          # one value per wavelength
print(J.shape, f"line-centre sensitivity {float(J[100]):+.2e} per km/s")

[spice] IcosphereModel constructed in 0.0 s


(200,) line-centre sensitivity +6.50e-03 per km/s


## A binary light curve with traced radii and inclination

In [4]:
from spice.models import Binary, find_binary_eclipses
from spice.models.binary import add_orbit, evaluate_orbit

N_VERT = 300
P = 0.01                                          # years
p1 = bb.to_parameters(dict(Teff=6000.))
p2 = bb.to_parameters(dict(Teff=4500.))
wl = jnp.linspace(4500., 7000., 300)

# Size the occlusion neighbour search once, at concrete radii, with headroom
# for the radii the fit will explore.
ref = Binary.from_bodies(IcosphereModel.construct(N_VERT, 1.0, 1.0, p1, bb.parameter_names),
                         IcosphereModel.construct(N_VERT, 0.8, 0.8, p2, bb.parameter_names))
NN = (int(1.5 * ref.n_neighbours1) + 1, int(1.5 * ref.n_neighbours2) + 1)

def make_binary(theta):
    R1, R2, incl_deg = theta
    b1 = IcosphereModel.construct(N_VERT, R1, 1.0, p1, bb.parameter_names)
    b2 = IcosphereModel.construct(N_VERT, R2, 0.8, p2, bb.parameter_names)
    return add_orbit(Binary.from_bodies(b1, b2, *NN), P, 0., 0., jnp.deg2rad(incl_deg),
                     0., 0., 0., 0., 0., 100)

def v_mag_binary(theta, t):
    m1, m2 = evaluate_orbit(make_binary(theta), t)
    f = sum(simulate_observed_flux(bb.intensity, m, jnp.log10(wl), disable_doppler_shift=True)[:, 0]
            for m in (m1, m2))
    return AB_passband_luminosity(V, wl, f)

theta0 = jnp.array([1.0, 0.8, 88.0])             # R1, R2 [R_sun], inclination [deg]
eclipses = find_binary_eclipses(make_binary(theta0))
for e in eclipses:
    print(f"{e['kind']} eclipse, front = {e['front']}: phase {e['T1']/P:.4f} - {e['T4']/P:.4f}")

[spice] IcosphereModel constructed in 9.2 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] Found 2 eclipse(s) in 0.8 s


partial eclipse, front = 2: phase 0.2270 - 0.2730
partial eclipse, front = 1: phase 0.7270 - 0.7730


In [5]:
t = 0.5 * (eclipses[0]['T1'] + eclipses[0]['mid'])  # half-way into the first eclipse
mag, J = v_mag_binary(theta0, t), jax.jacfwd(v_mag_binary)(theta0, t)
steps = jnp.array([0.005, 0.005, 0.05])
fd = [(v_mag_binary(theta0 + s * e, t) - v_mag_binary(theta0 - s * e, t)) / (2 * s)
      for s, e in zip(steps, jnp.eye(3))]
for name, ad, f in zip(["R1", "R2", "incl"], J, fd):
    print(f"dV/d{name:4s} autodiff {float(ad):+.5f}   finite difference {float(f):+.5f}")

[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.6 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


... (7 outputs left out)

[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


dV/dR1   autodiff -1.83212   finite difference -1.83211
dV/dR2   autodiff +0.29024   finite difference +0.29024
dV/dincl autodiff +0.05216   finite difference +0.05216


## Things that do not trace

In [6]:
for what, fn in [("jax.jit around a binary model", lambda: jax.jit(v_mag_binary)(theta0, t)),
                 ("jax.jit around a single-star magnitude", lambda: jax.jit(v_mag)(5777.)),
                 ("jax.jit around a single-star spectrum",
                  lambda: jax.jit(lambda v: normalised_profile(v))(20.)),
                 ("Binary.from_bodies with a traced radius",
                  lambda: jax.grad(lambda r: Binary.from_bodies(
                      IcosphereModel.construct(N_VERT, r, 1.0, p1, bb.parameter_names),
                      IcosphereModel.construct(N_VERT, 0.8, 0.8, p2, bb.parameter_names)).body1.radius)(1.0))]:
    try:
        fn()
        print(what, "-> works")
    except (jax.errors.ConcretizationTypeError, jax.errors.TracerArrayConversionError) as err:
        print(what, "->", type(err).__name__)

[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


jax.jit around a binary model -> ConcretizationTypeError
jax.jit around a single-star magnitude -> TracerArrayConversionError


jax.jit around a single-star spectrum -> works


[spice] IcosphereModel constructed in 0.4 s


[spice] IcosphereModel constructed in 0.0 s


Binary.from_bodies with a traced radius -> ConcretizationTypeError


## A gradient-based fit

In [7]:
truth = jnp.array([1.0, 0.8, 88.0])
times = np.concatenate([np.linspace(0.22, 0.28, 8), np.linspace(0.72, 0.78, 8)]) * P
sigma = 0.002                                     # 2 mmag
rng = np.random.default_rng(1)
data = (jnp.stack([v_mag_binary(truth, t) for t in times])
        + rng.normal(0., sigma, len(times)) + 0.37)  # + an unknown zero point

jac = jax.jacfwd(v_mag_binary)

def residuals(theta):
    """Residuals and Jacobian with the zero point solved analytically."""
    model = jnp.stack([v_mag_binary(theta, t) for t in times])
    J = jnp.stack([jac(theta, t) for t in times])
    r = data - model
    return r - r.mean(), J - J.mean(axis=0), float(r.mean())

# Levenberg-Marquardt on the exact Jacobian
theta = jnp.array([1.06, 0.72, 86.8])             # deliberately wrong start
lam = 1e-2
rc, Jc, zp = residuals(theta)
chi2 = float(jnp.sum((rc / sigma) ** 2))
start_model = data - rc                           # model + its best zero point
for it in range(6):
    A = Jc.T @ Jc
    step = jnp.linalg.solve(A + lam * jnp.diag(jnp.diag(A)), Jc.T @ rc)
    rc_new, Jc_new, zp_new = residuals(theta + step)
    chi2_new = float(jnp.sum((rc_new / sigma) ** 2))
    if chi2_new < chi2:                           # accept and trust the linearisation more
        theta, rc, Jc, zp, chi2, lam = theta + step, rc_new, Jc_new, zp_new, chi2_new, lam / 3
    else:                                         # reject and damp
        lam *= 5
    print(f"iteration {it}: chi2/N = {chi2 / len(times):8.3f}  theta = {np.round(np.asarray(theta), 4)}")

errors = np.sqrt(np.diag(np.linalg.inv(np.asarray(Jc.T @ Jc) / sigma ** 2)))
for name, v, e, tv in zip(["R1", "R2", "incl"], np.asarray(theta), errors, np.asarray(truth)):
    print(f"{name:4s} = {v:.4f} +- {e:.4f}   (true {tv})")
print(f"zero point {zp:.4f} (true 0.37)")

[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


... (473 outputs left out)

[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


iteration 5: chi2/N =    0.331  theta = [ 0.9933  0.8109 87.8864]


R1   = 0.9933 +- 0.0255   (true 1.0)
R2   = 0.8109 +- 0.0336   (true 0.8)
incl = 87.8864 +- 0.3335   (true 88.0)
zero point 0.3615 (true 0.37)


### The same gradient in an optax optimiser

In [8]:
import optax

scale = jnp.array([0.05, 0.05, 1.0])             # optimise z = (theta - start) / scale
start = jnp.array([1.06, 0.72, 86.8])
z = jnp.zeros(3)
opt = optax.adam(0.15)
state = opt.init(z)
for step in range(20):
    rc_k, Jc_k, _ = residuals(start + z * scale)
    if step % 5 == 0:
        print(f"Adam step {step:2d}: chi2/N = {float(jnp.sum((rc_k / sigma) ** 2)) / len(times):8.2f}")
    grad_z = -(rc_k / sigma ** 2) @ Jc_k * scale  # gradient of chi2 / 2 with respect to z
    updates, state = opt.update(grad_z, state)
    z = optax.apply_updates(z, updates)
rc_k, _, _ = residuals(start + z * scale)
print(f"Adam step 20: chi2/N = {float(jnp.sum((rc_k / sigma) ** 2)) / len(times):8.2f}")

[spice] IcosphereModel constructed in 0.2 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


... (1,337 outputs left out)

[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


[spice] IcosphereModel constructed in 0.0 s


Adam step 20: chi2/N =    52.70


In [9]:
final_model = data - rc                           # model + fitted zero point

def make_fig():
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
    for ax, sel, title in zip(axes, [slice(0, 8), slice(8, 16)], ["first eclipse", "second eclipse"]):
        ph = times[sel] / P
        ax.errorbar(ph, np.asarray(data)[sel], sigma, fmt="o", ms=4, label="synthetic data")
        ax.plot(ph, np.asarray(start_model)[sel], "--", label="start")
        ax.plot(ph, np.asarray(final_model)[sel], "-", label="Levenberg-Marquardt fit")
        ax.invert_yaxis(); ax.set_xlabel("orbital phase"); ax.set_title(title)
    axes[0].set_ylabel("V [mag]")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    return fig

save_doc_fig("gradient_fit", make_fig)
print(f"notebook run time {time.perf_counter() - T_START:.0f} s")

saved gradient_fit.png / gradient_fit_dark.png
notebook run time 316 s
